# Практическая работа №4: Временные ряды и пользовательские сессии
**Цель работы:** Научиться работать с датами и временем в pandas, выполнять агрегирование по временным интервалам и использовать оконные функции для анализа активности пользователей и их сессий на основе логов заказов и событий.

### Подготовка: Загрузка данных

In [ ]:
import pandas as pd

data_dir = 'data'

# Загружаем логи событий из events_lines.json (на основе данных из практики 3)
try:
    df_events = pd.read_json(f'{data_dir}/events_lines.json', lines=True)
    
    # Преобразуем текстовые даты в объекты datetime pandas
    df_events['event_time'] = pd.to_datetime(df_events['event_time'])
    
    # Извлекаем день недели (на английском: Monday, Tuesday...)
    df_events['day_of_week'] = df_events['event_time'].dt.day_name()
    
    print("Данные успешно загружены! Первые 3 строки:")
    display(df_events.head(3))
except FileNotFoundError:
    print("Файл events_lines.json не найден в папке data/")

### Задание 1. Построить по событиям перекрестную таблицу (pivot) по дням недели и типам событий.

In [ ]:
# Используем pd.crosstab для создания перекрестной таблицы (считаем количество событий)
pivot_events = pd.crosstab(index=df_events['day_of_week'], columns=df_events['event_type'])

# Для наглядности отсортируем дни недели в правильном календарном порядке
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
pivot_events = pivot_events.reindex(days_order)

print("Количество событий по дням недели и типам:")
display(pivot_events)

### Задание 2. Для каждого типа событий посчитать долю от общего количества событий в день.

In [ ]:
# Считаем общее количество событий за каждый день (сумма по строкам)
daily_totals = pivot_events.sum(axis=1)

# Делим каждое значение в строке на сумму этой строки (axis=0 - выравнивание по индексу дня недели)
pivot_proportions = pivot_events.div(daily_totals, axis=0)

print("Доля каждого события от общего количества в день (доли от 0 до 1):")
display(pivot_proportions.round(4))

print("\nТе же данные, но в процентах (%):")
display((pivot_proportions * 100).round(2))

### Задание 3. Найти день недели, где доля add_to_cart максимальна

In [ ]:
if 'add_to_cart' in pivot_proportions.columns:
    # Находим день недели (индекс) с максимальным значением в колонке 'add_to_cart'
    max_day = pivot_proportions['add_to_cart'].idxmax()
    
    # Достаем само максимальное значение для вывода
    max_value = pivot_proportions.loc[max_day, 'add_to_cart']
    
    print(f"⭐ День недели с максимальной долей 'add_to_cart': {max_day}")
    print(f"📊 Доля в этот день составляет: {max_value:.2%} ({max_value * 100:.2f}%)")
else:
    print("Событие 'add_to_cart' не найдено в данных.")